In [ ]:
import sys 
from pathlib import Path 
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer

from config import BERT_DIR, MODELS_DIR

CHECKPOINT_DIR = MODELS_DIR / "distilbert_results"

/Users/brandonwu/Documents/amazon_reviews_project/amazon/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
reviews_df = pd.read_parquet(BERT_DIR / "amazon_reviews_bert.parquet")

In [ ]:
train_df, test_df = train_test_split(
    reviews_df,
    test_size=0.2,
    random_state=42,
    stratify=reviews_df["rating"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.25,
    random_state=42,
    stratify=train_df["rating"]
)

In [4]:
# Conversion to Hugging Face datasets
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

In [5]:
# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [6]:
def tokenize_function(examples) -> DistilBertTokenizerFast:
    return tokenizer(
        examples["full_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

Map: 100%|██████████| 4029/4029 [00:00<00:00, 6714.73 examples/s]


In [7]:
# Remove raw text column and format for PyTorch
train_ds = train_ds.remove_columns(["full_text"])
val_ds = val_ds.remove_columns(["full_text"])
test_ds = test_ds.remove_columns(["full_text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [9]:
# Load DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Set training arguments
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7633.50it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# Create trainer, train, and evaluate
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

val_results = trainer.predict(val_ds)
test_results = trainer.predict(test_ds)

print("Validation metrics:", val_results.metrics)
print("Test metrics:", test_results.metrics)

/Users/brandonwu/Documents/amazon_reviews_project/amazon/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.145496,0.120540,0.965748,0.988528,0.891379,0.937443
2,0.059728,0.099418,0.975676,0.954623,0.961207,0.957904
3,0.030196,0.098191,0.977910,0.966057,0.956897,0.961455


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]
/Users/brandonwu/Documents/amazon_reviews_project/amazon/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.47it/s]
/Users/brandonwu/Documents/amazon_reviews_project/amazon/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.68it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.Lay

/Users/brandonwu/Documents/amazon_reviews_project/amazon/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Validation metrics: {'test_loss': 0.09819143265485764, 'test_accuracy': 0.9779101514023331, 'test_precision': 0.9660574412532638, 'test_recall': 0.9568965517241379, 'test_f1': 0.9614551754006063, 'test_runtime': 102.2624, 'test_samples_per_second': 39.399, 'test_steps_per_second': 2.464}
Test metrics: {'test_loss': 0.12474251538515091, 'test_accuracy': 0.9724497393894267, 'test_precision': 0.9549002601908065, 'test_recall': 0.9491379310344827, 'test_f1': 0.9520103761348897, 'test_runtime': 103.3662, 'test_samples_per_second': 38.978, 'test_steps_per_second': 2.438}


In [ ]:
save_dir = MODELS_DIR / "distilbert_binary_classifier"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)